## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import gc
import spatialdata_plot
from spatialdata.transformations import get_transformation, remove_transformation, set_transformation, Identity

matplotlib.use("Agg")
import matplotlib.patheffects as pe

import logging
logging.getLogger('ome_zarr').setLevel(logging.ERROR) # This mutes unnecessary Zarr warning (see: https://github.com/scverse/spatialdata/issues/400)

import sys
sys.path.append(os.path.dirname(os.path.abspath('')))
from utils.plotting import rel_fontsize

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/.venv_harpy_20Jan2026/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


## Specify metadata

In [ ]:
plots_base_dir = "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/plots/spatial_plots"

sample_metadata = [
    # Nova-ST
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "Exp93-sampleSPC022_100M",
        "technology": "Nova-ST",
        "microns_per_pixel": 1,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": False,
        "image_layer": "Exp93-sampleSPC022_100M_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC002/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "Exp93-sampleSPC002_100M",
        "technology": "Nova-ST",
        "microns_per_pixel": 1,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": False,
        "image_layer": "Exp93-sampleSPC002_100M_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp100-sampleSPC004/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "Exp100-sampleSPC004_100M",
        "technology": "Nova-ST",
        "microns_per_pixel": 1,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": False,
        "image_layer": "Exp100-sampleSPC004_100M_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp65-sampleSPC014/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "Exp65-sampleSPC014_100M",
        "technology": "Nova-ST",
        "microns_per_pixel": 1,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "Exp65-sampleSPC014_100M_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp65-sampleSPC014/subsampled_100M/NovaScope/novascope_output/seq1st/NovaSeq6000_20241107/nbcds/03D1/1_1.sbcds.sorted.tsv.gz"
    },
        {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp100-samplePSB/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "Exp100-samplePSB_100M",
        "technology": "Nova-ST",
        "microns_per_pixel": 1,
        "sample_type": "Rice root",
        "replicate": None,
        "grey": True,
        "image_layer": "Exp100-samplePSB_100M_CFW",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },

    # Stereo-Seq
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/spc002/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "spc002",
        "technology": "Stereo-Seq",
        "microns_per_pixel": 0.5,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": True,
        "image_layer": "spc002_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/spc004/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "spc004",
        "technology": "Stereo-Seq",
        "microns_per_pixel": 0.5,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": True,
        "image_layer": "spc004_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/spc014/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "spc014",
        "technology": "Stereo-Seq",
        "microns_per_pixel": 0.5,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": True,
        "image_layer": "spc014_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/raw/Stereo-Seq/spc014/mask/Y40048L4.barcodeToPos.h5"
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/spc022/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "spc022",
        "technology": "Stereo-Seq",
        "microns_per_pixel": 0.5,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": True,
        "image_layer": "spc022_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Stereo-Seq/psb/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "psb",
        "technology": "Stereo-Seq",
        "microns_per_pixel": 0.5,
        "sample_type": "Rice root",
        "replicate": None,
        "grey": True,
        "image_layer": "psb_CFW",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },

    # Curio Seeker
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC11/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SC11",
        "technology": "Curio Seeker",
        "microns_per_pixel": 0.641,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": None,
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/raw/Curio_seeker/barcode_decoding_files/A0080_051_BeadBarcodes.txt"

    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC13/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SC13",
        "technology": "Curio Seeker",
        "microns_per_pixel": 0.637,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": False,
        "image_layer": None,
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC12/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SC12",
        "technology": "Curio Seeker",
        "microns_per_pixel": 0.637,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": False,
        "image_layer": None,
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SC14",
        "technology": "Curio Seeker",
        "microns_per_pixel": 0.635,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": False,
        "image_layer": None,
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },

    # Visium (FFPE)
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA013",
        "technology": "Visium (FFPE)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA013_full_image",
        "table_layer": "table",
        "labels_layer": "SCA013_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA014/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA014",
        "technology": "Visium (FFPE)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA014_full_image",
        "table_layer": "table",
        "labels_layer": "SCA014_labels",
        
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA016/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA016",
        "technology": "Visium (FFPE)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA016_full_image",
        "table_layer": "table",
        "labels_layer": "SCA016_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA016/subsampled_100M/SpaceRanger/outs/spatial/tissue_positions.csv"
        
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA015/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA015",
        "technology": "Visium (FFPE)",
        "microns_per_pixel": 0.442,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA015_full_image",
        "table_layer": "table",
        "labels_layer": "SCA015_labels",
    },    

    # Visium HD (FFPE)
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA001/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA001",
        "technology": "Visium HD (FFPE)",
        "microns_per_pixel": 0.220,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA001_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA002/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA002",
        "technology": "Visium HD (FFPE)",
        "microns_per_pixel": 0.220,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA002_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA003",
        "technology": "Visium HD (FFPE)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA003_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA003/subsampled_100M/SpaceRanger/outs/binned_outputs/square_002um/spatial/tissue_positions.parquet"
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA010/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA010",
        "technology": "Visium HD (FFPE)",
        "microns_per_pixel": 0.442,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA010_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    
    # Visium HD (FF)
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA017/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA017",
        "technology": "Visium HD (FF)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA017_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA018/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA018",
        "technology": "Visium HD (FF)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA018_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA019/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA019",
        "technology": "Visium HD (FF)",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA019_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA019/subsampled_100M/SpaceRanger/outs/binned_outputs/square_002um/spatial/tissue_positions.parquet"
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA020/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA020",
        "technology": "Visium HD (FF)",
        "microns_per_pixel": 0.442,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA020_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },

    # Visium HD 3'
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA021/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA021",
        "technology": "Visium HD 3'",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse liver",
        "replicate": 1,
        "grey": False,
        "image_layer": "SCA021_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA023/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA023",
        "technology": "Visium HD 3'",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse liver",
        "replicate": 2,
        "grey": False,
        "image_layer": "SCA023_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA024/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "SCA024",
        "technology": "Visium HD 3'",
        "microns_per_pixel": 0.442,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "SCA024_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/SCA024/subsampled_100M/SpaceRanger/outs/binned_outputs/square_002um/spatial/tissue_positions.parquet"
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/FPE010/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "FPE010",
        "technology": "Visium HD 3'",
        "microns_per_pixel": 0.650,
        "sample_type": "Rice root",
        "replicate": None,
        "grey": True,
        "image_layer": "FPE010_full_image",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },

    # BMKMANU S1000
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "exp74",
        "technology": "BMKMANU S1000",
        "microns_per_pixel": 0.341,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "exp74_he",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S1000/exp74/subsampled_100M/BSTMAtrix/exp74_100M_11Aug2025/barcode_pos.tsv"
    },

    # BMKMANU S3000
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "exp101",
        "technology": "BMKMANU S3000",
        "microns_per_pixel": 0.340,
        "sample_type": "Mouse lung",
        "replicate": None,
        "grey": False,
        "image_layer": "exp101_he",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "exp100",
        "technology": "BMKMANU S3000",
        "microns_per_pixel": 0.340,
        "sample_type": "Mouse liver",
        "replicate": None,
        "grey": False,
        "image_layer": "exp100_he",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "exp99",
        "technology": "BMKMANU S3000",
        "microns_per_pixel": 0.340,
        "sample_type": "Mouse brain",
        "replicate": None,
        "grey": False,
        "image_layer": "exp99_he",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
        "barcode_file": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/subsampled_100M/BSTMAtrix/exp99_100M_11Aug2025/barcode_pos.tsv"
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp102/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "exp102",
        "technology": "BMKMANU S3000",
        "microns_per_pixel": 0.340,
        "sample_type": "Human lung",
        "replicate": None,
        "grey": False,
        "image_layer": "exp102_he",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
    {
        "sdata_path": "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/psb/subsampled_100M/harpy/sdata.zarr",
        "dataset_id": "psb",
        "technology": "BMKMANU S3000",
        "microns_per_pixel": 0.340,
        "sample_type": "Rice root",
        "replicate": None,
        "grey": True,
        "image_layer": "psb_CFW",
        "table_layer": "hexagonal_grid_20um_table",
        "labels_layer": "hexagonal_grid_20um_labels",
    },
]

tech_levels = (
    "BMKMANU S1000"
    "BMKMANU S3000",
    "Curio Seeker",
    "Nova-ST",
    "Stereo-Seq",
    "Visium (FFPE)",
    "Visium HD (FFPE)",
    "Visium HD (FF)",
    "Visium HD 3'",
)


In [ ]:
for sample in sample_metadata:
    sdata_path = sample["sdata_path"]
    dataset_id = sample["dataset_id"]
    technology = sample["technology"]
    microns_per_pixel = sample["microns_per_pixel"]
    sample_type = sample["sample_type"]
    replicate = sample["replicate"]
    grey = sample["grey"]
    image_layer = sample["image_layer"]
    table_layer = sample["table_layer"]
    labels_layer = sample["labels_layer"]

    # Read sdata
    sdata = sd.read_zarr(sdata_path)

    # Fix shapes layer for plotting
    if technology != "Curio Seeker":
        gdf = sdata.shapes[f"{dataset_id}_tissue"]
        gdf = gdf.dissolve()
        sdata = hp.sh.add_shapes_layer(
            sdata,
            input=gdf,
            output_layer=f"{dataset_id}_tissue",
            transformations=get_transformation(
                sdata.shapes[f"{dataset_id}_tissue"], get_all=True
            ),
            overwrite=True,
        )

    # Specify plotting parameters
    if replicate:
        plots_path = os.path.join(
            plots_base_dir, sample_type, f"{technology}_{replicate}"
        )
    else:
        plots_path = os.path.join(plots_base_dir, sample_type, technology)

    os.makedirs(plots_path, exist_ok=True)

    if grey:
        cmap = "gray"
        p99 = (
            sdata.images[image_layer]["scale0"]["image"]
            .stack(pixels=sdata.images[image_layer]["scale0"]["image"].dims)
            .quantile(0.99, dim="pixels")
            .compute()
            .item()
        )
        norm_CFW = matplotlib.colors.Normalize(vmin=0, vmax=p99)
        scale_bar_color = "white"
                
        if sample_type == "Rice root":
            if technology == "Visium HD 3'":
                norm_CFW = matplotlib.colors.Normalize(vmin=0, vmax=500)
            if technology == "BMKMANU S3000":
                norm_CFW = matplotlib.colors.Normalize(vmin=0, vmax=6000)
            if technology == "Stereo-Seq":
                norm_CFW = matplotlib.colors.Normalize(vmin=0, vmax=1500)
            if technology == "Nova-ST":
                norm_CFW = matplotlib.colors.Normalize(vmin=0, vmax=2000)
                
    else:
        cmap = None
        norm_CFW = None
        scale_bar_color = "black"


    if technology != "Curio Seeker":
        width_px = sdata.images[image_layer]["scale0"]["image"].shape[2]
        height_px = sdata.images[image_layer]["scale0"]["image"].shape[1]
    elif technology == "Curio Seeker":
        width_px = sdata.labels[f"{dataset_id}_labels"].shape[1]
        height_px = sdata.labels[f"{dataset_id}_labels"].shape[0]

    width_um = width_px * microns_per_pixel
    height_um = height_px * microns_per_pixel

    um_per_inch = 2000
    fig_width = width_um / um_per_inch
    fig_height = height_um / um_per_inch
    figsize = (fig_width, fig_height)

    scale_bar_um = 1000  # 1 mm scale bar
    scale_bar_px = scale_bar_um / microns_per_pixel
    DPI = 300

    # Plot microscopy image
    if technology != "Curio Seeker":
        fig = plt.figure(figsize=figsize, dpi=DPI)
        fig.set_constrained_layout(False)
        ax = fig.add_axes([0, 0, 1, 1])

        sdata.pl.render_images(
            element=image_layer, scale="full", cmap=cmap, norm=norm_CFW,
        ).pl.render_shapes(
            element=f"{dataset_id}_tissue", color="red", fill_alpha=0, outline_width=0.7, outline_alpha=1, outline_color="red",
        ).pl.render_shapes(
            element=f"{dataset_id}_capture_area", color="#00BFFF", fill_alpha=0, outline_width=2.2, outline_alpha=1, outline_color="#00BFFF",
        ).pl.show(
            coordinate_systems=dataset_id,
            title=None,
            colorbar=False,
            ax=ax,
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")
        ax.set_frame_on(False)

        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        xmin, xmax = min(x0, x1), max(x0, x1)
        ymin, ymax = min(y0, y1), max(y0, y1)

        extent_width_px = xmax - xmin
        extent_height_px = ymax - ymin
        extent_width_um = extent_width_px * microns_per_pixel
        extent_height_um = extent_height_px * microns_per_pixel

        um_per_inch = 2000
        fig_width = extent_width_um / um_per_inch
        fig_height = extent_height_um / um_per_inch
        fig.set_size_inches(fig_width, fig_height, forward=True)

        offset_um = 500
        offset_px = offset_um / microns_per_pixel
        scale_bar_x = xmin + offset_px
        scale_bar_y = ymax - offset_px

        ax.plot(
            [scale_bar_x, scale_bar_x + scale_bar_px],
            [scale_bar_y, scale_bar_y],
            color=scale_bar_color,
            linewidth=7,
            solid_capstyle="butt",
        )

        save_path = os.path.join(plots_path, "Microscopy.png")
        fig.savefig(save_path, facecolor="white", transparent=False, dpi=DPI)
        plt.close(fig)
        print("Plot saved at:", save_path)

    # Plot total counts and genes
    qc_metrics = {
        "n_genes_by_counts": "Number of genes",
        "total_counts": "unique UMI counts",
    }

    for qc_metric, qc_label in qc_metrics.items():
        gc.collect()

        values = sdata.tables[table_layer].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)

        fig = plt.figure(figsize=figsize, dpi=DPI)
        fig.set_constrained_layout(False)
        ax = fig.add_axes([0, 0, 0.8, 1])

        if technology != "Curio Seeker":
            sdata.pl.render_images(
                element=image_layer, scale="full", cmap=cmap, norm=norm_CFW, alpha=0
            ).pl.render_shapes(
                element=f"{dataset_id}_tissue", color="red", fill_alpha=0, outline_width=0.7, outline_alpha=1, outline_color="red",
            ).pl.show(
                coordinate_systems=dataset_id,
                title=None,
                colorbar=False,
                ax=ax,
            )

        sdata.pl.render_labels(
            element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, scale ="full", color=qc_metric, table_name=table_layer, cmap="viridis", norm=norm, colorbar=True,
        ).pl.render_shapes(
            element=f"{dataset_id}_capture_area", color="magenta", fill_alpha=0, outline_width=2.2, outline_alpha=1, outline_color="magenta",
        ).pl.show(
            coordinate_systems=dataset_id,
            title=None,
            ax=ax,
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_title('')
        ax.set_frame_on(False)

        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()

        xmin, xmax = min(x0, x1), max(x0, x1)
        ymin, ymax = min(y0, y1), max(y0, y1)

        extent_width_px = xmax - xmin
        extent_height_px = ymax - ymin

        extent_width_um = extent_width_px * microns_per_pixel
        extent_height_um = extent_height_px * microns_per_pixel

        um_per_inch = 2000

        data_fig_width = extent_width_um / um_per_inch
        data_fig_height = extent_height_um / um_per_inch

        cbar_width_in = 0.28
        cbar_pad_in = 0.10
        cbar_label_pad_in = 0.35

        total_fig_width = data_fig_width + cbar_pad_in + cbar_width_in + cbar_label_pad_in

        fig.set_size_inches(total_fig_width, data_fig_height, forward=True)

        ax_width_frac = data_fig_width / total_fig_width
        ax.set_position([0, 0, ax_width_frac, 1])

        offset_um = 500
        offset_px = offset_um / microns_per_pixel

        scale_bar_x = xmin + offset_px
        scale_bar_y = ymax - offset_px

        ax.plot(
            [scale_bar_x, scale_bar_x + scale_bar_px],
            [scale_bar_y, scale_bar_y],
            color="black",
            linewidth=7,
            solid_capstyle="butt"
        )

        other_axes = [a for a in fig.axes if a is not ax]
        if other_axes:
            cbar = other_axes[-1]
            cbar_left = (data_fig_width + cbar_pad_in) / total_fig_width
            cbar_width_frac = cbar_width_in / total_fig_width
            cbar.set_position([cbar_left, 0.08, cbar_width_frac, 0.84])
            cbar.set_ylabel(qc_label, rotation=270, labelpad=20, fontsize=10)
            cbar.tick_params(axis="y", which="both", labelsize=10)

        save_path = os.path.join(plots_path, f'{qc_metric}.png')
        fig.savefig(save_path, facecolor="white", transparent=False, dpi=DPI, bbox_inches="tight")
        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

In [ ]:
from matplotlib.patches import Rectangle
from shapely.geometry import Point, box
from shapely.affinity import translate as shapely_translate
import h5py

for sample in sample_metadata:
    sdata_path = sample["sdata_path"]
    dataset_id = sample["dataset_id"]
    technology = sample["technology"]
    microns_per_pixel = sample["microns_per_pixel"]
    sample_type = sample["sample_type"]
    image_layer = sample["image_layer"]
    labels_layer = sample["labels_layer"]

    if sample_type != "Mouse brain":
        continue

    barcode_file = sample["barcode_file"]

    # Specify shapes layer and spot size
    if technology == "Visium (FFPE)":
        table_layer = "table"
        shapes_layer = dataset_id
        spot_size_um = 55
    elif technology == "Curio Seeker":
        table_layer = "table"
        shapes_layer = dataset_id
    elif technology in ["Visium HD (FFPE)", "Visium HD (FF)", "Visium HD 3'"]:
        table_layer = "square_002um"
        shapes_layer = f"{dataset_id}_square_002um"
        spot_size_um = 2
    elif technology in ["BMKMANU S1000", "BMKMANU S3000"]:
        table_layer = f"{dataset_id}_level_1_table"
        shapes_layer = f"{dataset_id}_level_1_shapes"
        spot_size_um = 2.5
    elif technology == "Nova-ST":
        table_layer = "table"
        shapes_layer = f"{dataset_id}_shapes"
        spot_size_um = 0.3
    elif technology == "Stereo-Seq":
        table_layer = f"{dataset_id}_raw_table"
        shapes_layer = f"{dataset_id}_raw_shapes"
        spot_size_um = 0.22

    # Read sdata
    sdata = sd.read_zarr(sdata_path)

    if technology == "Curio Seeker":
        min_x = sdata.labels[f"{dataset_id}_labels"].shape[0] / 2 # Center of image
        max_x = min_x + round(120 / microns_per_pixel) # 120 micrometer
        min_y = sdata.labels[f"{dataset_id}_labels"].shape[1] / 2 # Center of image
        max_y = min_y + round(120 / microns_per_pixel) # 120 micrometer
    elif technology == "Nova-ST":
        min_x = 6090
        max_x = min_x + round(120 / microns_per_pixel) # 120 micrometer
        min_y = 2960
        max_y = min_y + round(120 / microns_per_pixel) # 120 micrometer
    elif technology == "BMKMANU S1000":
        min_x = 9700
        max_x = min_x + round(120 / microns_per_pixel) # 120 micrometer
        min_y = 10300
        max_y = min_y + round(120 / microns_per_pixel) # 120 micrometer
    elif technology == "BMKMANU S3000":
        min_x = 10300
        max_x = min_x + round(120 / microns_per_pixel) # 120 micrometer
        min_y = 10100
        max_y = min_y + round(120 / microns_per_pixel) # 120 micrometer
    elif technology == "Stereo-Seq":
        min_x = 11900
        max_x = min_x + round(120 / microns_per_pixel) # 120 micrometer
        min_y = 11900
        max_y = min_y + round(120 / microns_per_pixel) # 120 micrometer
    else:
        min_x = sdata.images[image_layer]['scale0']['image'].shape[2] / 2 # Center of image
        max_x = min_x + round(120 / microns_per_pixel) # 120 micrometer
        min_y = sdata.images[image_layer]['scale0']['image'].shape[1] / 2 # Center of image
        max_y = min_y + round(120 / microns_per_pixel) # 120 micrometer

    sdata_crop = sdata.query.bounding_box(
        min_coordinate=[min_x, min_y], 
        max_coordinate=[max_x, max_y], 
        axes=("x", "y"), 
        target_coordinate_system=dataset_id
    )

    # Add barcode_positions shapes layer
    barcode_positions_layer = "barcode_positions"

    if technology in ["Visium (FFPE)", "Visium HD (FFPE)", "Visium HD (FF)", "Visium HD 3'"]:
        if technology == "Visium (FFPE)":
            barcode_df = pd.read_csv(barcode_file)
            margin_px = (spot_size_um / microns_per_pixel)
        else:
            barcode_df = pd.read_parquet(barcode_file)
            margin_px = (spot_size_um / microns_per_pixel)

        barcode_df_crop = barcode_df[
            (barcode_df["pxl_col_in_fullres"] >= min_x - margin_px) &
            (barcode_df["pxl_col_in_fullres"] <= max_x + margin_px) &
            (barcode_df["pxl_row_in_fullres"] >= min_y - margin_px) &
            (barcode_df["pxl_row_in_fullres"] <= max_y + margin_px)
        ].copy()
        
        if technology == "Visium (FFPE)":
            spot_radius_px = (spot_size_um / microns_per_pixel) / 2  # 55 um diameter
            barcode_df_crop["geometry"] = [
                Point(x, y).buffer(spot_radius_px)
                for x, y in zip(
                    barcode_df_crop["pxl_col_in_fullres"],
                    barcode_df_crop["pxl_row_in_fullres"]
                )
            ]
        else:
            side_px = spot_size_um / microns_per_pixel  # 2 x 2 um squares
            half_side_px = side_px / 2
            barcode_df_crop["geometry"] = [
                box(x - half_side_px, y - half_side_px, x + half_side_px, y + half_side_px)
                for x, y in zip(
                    barcode_df_crop["pxl_col_in_fullres"],
                    barcode_df_crop["pxl_row_in_fullres"]
                )
            ]
        
        barcode_gdf = gpd.GeoDataFrame(barcode_df_crop, geometry="geometry")

        crop_box = box(min_x, min_y, max_x, max_y)
        barcode_gdf = barcode_gdf[barcode_gdf.geometry.intersects(crop_box)].copy()

        sdata_crop = hp.sh.add_shapes_layer(
            sdata_crop,
            input=barcode_gdf,
            output_layer=barcode_positions_layer,
            transformations={dataset_id: Identity()},
            overwrite=True,
        )

    elif technology == "Nova-ST":
        barcode_df = pd.read_csv(barcode_file, delimiter='\t', header=None, names=['barcode', 'lane_id', 'tile_id', 'x', 'y', 'count'])
        margin_px = (1 / microns_per_pixel)

        barcode_df["x"] = barcode_df["x"] / 1000
        barcode_df["y"] = barcode_df["y"] / 1000

        barcode_df_crop = barcode_df[
            (barcode_df["x"] >= min_x - margin_px) &
            (barcode_df["x"] <= max_x + margin_px) &
            (barcode_df["y"] >= min_y - margin_px) &
            (barcode_df["y"] <= max_y + margin_px)
        ].copy()

        spot_radius_px = (spot_size_um / microns_per_pixel) / 2  # 0.3 um diameter
        barcode_df_crop["geometry"] = [
            Point(x, y).buffer(spot_radius_px)
            for x, y in zip(
                barcode_df_crop["x"],
                barcode_df_crop["y"]
            )
        ]

        barcode_gdf = gpd.GeoDataFrame(barcode_df_crop, geometry="geometry")

        crop_box = box(min_x, min_y, max_x, max_y)
        barcode_gdf = barcode_gdf[barcode_gdf.geometry.intersects(crop_box)].copy()

        sdata_crop = hp.sh.add_shapes_layer(
            sdata_crop,
            input=barcode_gdf,
            output_layer=barcode_positions_layer,
            transformations={dataset_id: Identity()},
            overwrite=True,
        )

    elif technology == "Stereo-Seq":
        with h5py.File(barcode_file, "r") as f:
            dset = f["bpMatrix_1"]

            slab = dset[min_y:max_y, min_x:max_x, 0]
            ys, xs = np.nonzero(slab)

        barcode_df_crop = pd.DataFrame({
            "x": xs + min_x,
            "y": ys + min_y,
        })

        margin_px = (1 / microns_per_pixel)

        spot_radius_px = (spot_size_um / microns_per_pixel) / 2  # 0.22 um diameter
        barcode_df_crop["geometry"] = [
            Point(x, y).buffer(spot_radius_px)
            for x, y in zip(
                barcode_df_crop["x"],
                barcode_df_crop["y"]
            )
        ]

        barcode_gdf = gpd.GeoDataFrame(barcode_df_crop, geometry="geometry")

        crop_box = box(min_x, min_y, max_x, max_y)
        barcode_gdf = barcode_gdf[barcode_gdf.geometry.intersects(crop_box)].copy()

        sdata_crop = hp.sh.add_shapes_layer(
            sdata_crop,
            input=barcode_gdf,
            output_layer=barcode_positions_layer,
            transformations={dataset_id: Identity()},
            overwrite=True,
        )

    elif technology == "Curio Seeker":
        barcode_df = pd.read_csv(barcode_file, delimiter='\t', header=None, names=['barcode', 'y', 'x'])
        minx_geom, miny_geom, maxx_geom, maxy_geom = sdata.shapes[dataset_id].geometry.iloc[0].bounds
        spot_radius_px = (maxx_geom - minx_geom) / 2
        spot_size_um = spot_radius_px * microns_per_pixel * 2

        barcode_df["x"] = - barcode_df["x"]

        offset_x = -barcode_df['x'].min() + spot_radius_px
        offset_y = -barcode_df['y'].min() + spot_radius_px

        margin_px = spot_radius_px

        barcode_df_crop = barcode_df[
            (barcode_df["x"] + offset_x >= min_x - margin_px) &
            (barcode_df["x"] + offset_x <= max_x + margin_px) &
            (barcode_df["y"] + offset_y >= min_y - margin_px) &
            (barcode_df["y"] + offset_y <= max_y + margin_px)
        ].copy()

        barcode_df_crop["geometry"] = [
            shapely_translate(Point(x, y).buffer(spot_radius_px), xoff=offset_x, yoff=offset_y)
            for x, y in zip(
                barcode_df_crop["x"],
                barcode_df_crop["y"]
            )
        ]

        barcode_gdf = gpd.GeoDataFrame(barcode_df_crop, geometry="geometry")

        crop_box = box(min_x, min_y, max_x, max_y)
        barcode_gdf = barcode_gdf[barcode_gdf.geometry.intersects(crop_box)].copy()

        sdata_crop = hp.sh.add_shapes_layer(
            sdata_crop,
            input=barcode_gdf,
            output_layer=barcode_positions_layer,
            transformations={dataset_id: Identity()},
            overwrite=True,
        )
    
    elif technology in ["BMKMANU S1000", "BMKMANU S3000"]:
        barcode_positions_layer = shapes_layer # Already contains all spots in capture area

    # Spots with at least 1 UMI
    barcode_UMI_df = pd.DataFrame({
        "x": sdata.tables[table_layer].obsm['spatial'][:, 0],
        "y": sdata.tables[table_layer].obsm['spatial'][:, 1],
        "total_counts": sdata.tables[table_layer].obs["total_counts"].values
    })

    barcode_UMI_df = barcode_UMI_df[barcode_UMI_df["total_counts"] > 0]

    margin_px = (spot_size_um / microns_per_pixel)

    barcode_UMI_df_crop = barcode_UMI_df[
        (barcode_UMI_df["x"] >= min_x - margin_px) &
        (barcode_UMI_df["x"] <= max_x + margin_px) &
        (barcode_UMI_df["y"] >= min_y - margin_px) &
        (barcode_UMI_df["y"] <= max_y + margin_px)
    ].copy()

    if technology in ["Visium HD (FFPE)", "Visium HD (FF)", "Visium HD 3'"]:
        side_px = spot_size_um / microns_per_pixel  # 2 x 2 um squares
        half_side_px = side_px / 2
        barcode_UMI_df_crop["geometry"] = [
            box(x - half_side_px, y - half_side_px, x + half_side_px, y + half_side_px)
            for x, y in zip(
                barcode_UMI_df_crop["x"],
                barcode_UMI_df_crop["y"]
            )
        ]

    else:
        spot_radius_px = (spot_size_um / microns_per_pixel) / 2
        barcode_UMI_df_crop["geometry"] = [
            Point(x, y).buffer(spot_radius_px)
            for x, y in zip(
                barcode_UMI_df_crop["x"],
                barcode_UMI_df_crop["y"]
            )
        ]

    barcode_UMI_gdf = gpd.GeoDataFrame(barcode_UMI_df_crop, geometry="geometry")

    crop_box = box(min_x, min_y, max_x, max_y)
    barcode_UMI_gdf = barcode_UMI_gdf[barcode_UMI_gdf.geometry.intersects(crop_box)].copy()

    sdata_crop = hp.sh.add_shapes_layer(
        sdata_crop,
        input=barcode_UMI_gdf,
        output_layer="spots_UMI",
        transformations={dataset_id: Identity()},
        overwrite=True,
    )

    # Specify plotting parameters
    plots_path = os.path.join(plots_base_dir, sample_type, technology)
    os.makedirs(plots_path, exist_ok=True)

    figsize = (5, 5)

    scale_bar_um = 20  # 20 um scale bar
    scale_bar_px = scale_bar_um / microns_per_pixel
    DPI = 300

    # Plot spots
    fig = plt.figure(figsize=figsize, dpi=DPI)
    fig.set_constrained_layout(False)
    ax = fig.add_axes([0, 0, 1, 1])

    sdata_crop.pl.render_shapes(
        element=barcode_positions_layer, fill_alpha=0.15, outline_width=0.55, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.render_shapes(
        element="spots_UMI", fill_alpha=0.35, outline_width=0.65, outline_alpha=1, color='orange', outline_color='orange', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=None,
        colorbar=False,
        ax=ax,
        figsize=figsize
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    offset_um = 10
    offset_px = offset_um / microns_per_pixel
    scale_bar_x = min_x + offset_px
    scale_bar_y = max_y - offset_px

    ax.plot(
        [scale_bar_x, scale_bar_x + scale_bar_px],
        [scale_bar_y, scale_bar_y],
        color="black",
        linewidth=5,
        solid_capstyle="butt",
    )

    save_path = os.path.join(plots_path, "Spots.png")
    fig.savefig(save_path, facecolor="white", transparent=False, dpi=DPI)
    plt.close(fig)
    print("Plot saved at:", save_path)